In [13]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [15]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
llm = ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [17]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents, embedding=embeddings)

In [18]:
vectorstore.similarity_search("cat")

[Document(id='c464d95a-8ee9-4678-bea6-4cf61e609f5b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='aa23b84a-b322-4d47-a347-96535a658f9f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='ec4b5e3d-d552-4713-8e35-7e42d932d2b0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='acfefb4f-3aae-46f1-b52a-4453602f6510', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [19]:
## Async query
await vectorstore.asimilarity_search("cat")

[Document(id='c464d95a-8ee9-4678-bea6-4cf61e609f5b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='aa23b84a-b322-4d47-a347-96535a658f9f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='ec4b5e3d-d552-4713-8e35-7e42d932d2b0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
 Document(id='acfefb4f-3aae-46f1-b52a-4453602f6510', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [20]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='c464d95a-8ee9-4678-bea6-4cf61e609f5b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='aa23b84a-b322-4d47-a347-96535a658f9f', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
  0.9351056814193726),
 (Document(id='ec4b5e3d-d552-4713-8e35-7e42d932d2b0', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706),
 (Document(id='acfefb4f-3aae-46f1-b52a-4453602f6510', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.'),
  1.574089765548706)]

### Retrievers
LangChain VectorStore objects do not subclass Runnable, and so cannot immediately be integrated into LangChain Expression Language chains.

LangChain Retrievers are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations) and are designed to be incorporated in LCEL chains.

In [21]:
# Converts the vector store into a Retriever runnable
# search_type="similarity" Performs cosine / dot-product similarity search
# k = number of documents to return per query
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat", "dog"])

[[Document(id='c464d95a-8ee9-4678-bea6-4cf61e609f5b', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='acfefb4f-3aae-46f1-b52a-4453602f6510', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

<context>
{context}
</context>

"""

prompt = ChatPromptTemplate.from_messages([("human", message)])
rag_chain = {
    "context" : retriever,
    "question" : RunnablePassthrough()
} | prompt | llm

response = rag_chain.invoke("tell me about dogs")
response.content


'Dogs are great companions, known for their loyalty and friendliness.'